# 10 — MS-TCN-style Baseline on Hidden ProcedureVRL Embeddings

This notebook trains and evaluates a **visual-only TAS baseline** on the hidden ProcedureVRL video embeddings extracted in notebook 09.

Input features from notebook 09:

```text
per video: [512, 16]
```

These are the hidden 512-dimensional video embeddings captured from:

```text
model.head = Linear(768 -> 512)
```

This is stronger than the previous coarse ProcedureVRL output representation:

```text
coarse ProcedureVRL outputs: [9871, 16]
hidden ProcedureVRL embeddings: [512, 16]
```

Goal of this notebook:

```text
hidden ProcedureVRL features
→ MS-TCN-style visual-only baseline
→ Acc / Edit / F1@10 / F1@25 / F1@50
```

After this notebook finishes, we should have a clean ProcedureVRL-hidden visual-only baseline. The next notebook will be the text-teacher / video-only-student KD experiment on the same hidden ProcedureVRL feature space.

## 1. Mount Drive and imports

In [16]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

from pathlib import Path
import os
import sys
import json
import time
import random
import shutil
from collections import defaultdict

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

Mounted at /content/drive
Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
PyTorch: 2.11.0+cu128
CUDA: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


## 2. Configuration

In [17]:
DRIVE_ROOT = Path("/content/drive/MyDrive/mmf_tas_lab_data")

# Output of notebook 09.
HIDDEN_RUN_ROOT = (
    DRIVE_ROOT
    / "text_assisted_tas"
    / "breakfast"
    / "procedurevrl_hidden"
    / "runs"
    / "procedurevrl_hidden_full_split1_split1_views16"
)

DATA_ROOT = HIDDEN_RUN_ROOT / "mstcn_format"
FEATURE_DIR = DATA_ROOT / "features"
GT_DIR = DATA_ROOT / "groundTruth"
SPLIT_DIR = DATA_ROOT / "splits"
MAPPING_PATH = DATA_ROOT / "mapping.txt"

# This notebook output.
OUT_ROOT = (
    DRIVE_ROOT
    / "text_assisted_tas"
    / "breakfast"
    / "procedurevrl_hidden"
    / "runs"
    / "mstcn_hidden_baseline_split1"
)
MODEL_DIR = OUT_ROOT / "models"
PRED_DIR = OUT_ROOT / "predictions"
RESULTS_DIR = OUT_ROOT / "results"

for p in [OUT_ROOT, MODEL_DIR, PRED_DIR, RESULTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

SPLIT_ID = 1
TRAIN_SPLIT = SPLIT_DIR / f"train.split{SPLIT_ID}.bundle"
TEST_SPLIT = SPLIT_DIR / f"test.split{SPLIT_ID}.bundle"

# Training config.
SEED = 7
EPOCHS = 120
BATCH_SIZE = 64
NUM_WORKERS = 0
LEARNING_RATE = 5e-4
WEIGHT_DECAY = 1e-4
GRAD_CLIP = 5.0
EVAL_EVERY = 5

# MS-TCN-style architecture.
NUM_STAGES = 4
NUM_LAYERS = 6
NUM_F_MAPS = 64
DROPOUT = 0.5
SMOOTHING_LOSS_WEIGHT = 0.15

# Normalize hidden embeddings using train split statistics.
STANDARDIZE_FEATURES = True

# Segmental metrics usually ignore background/silence classes if present.
CANDIDATE_IGNORE_CLASSES = ["background", "SIL", "silence"]

required = [HIDDEN_RUN_ROOT, DATA_ROOT, FEATURE_DIR, GT_DIR, SPLIT_DIR, MAPPING_PATH, TRAIN_SPLIT, TEST_SPLIT]
for p in required:
    print(p, "->", p.exists())
    assert p.exists(), f"Missing required path: {p}"

print("OUT_ROOT:", OUT_ROOT)

/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/procedurevrl_hidden/runs/procedurevrl_hidden_full_split1_split1_views16 -> True
/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/procedurevrl_hidden/runs/procedurevrl_hidden_full_split1_split1_views16/mstcn_format -> True
/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/procedurevrl_hidden/runs/procedurevrl_hidden_full_split1_split1_views16/mstcn_format/features -> True
/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/procedurevrl_hidden/runs/procedurevrl_hidden_full_split1_split1_views16/mstcn_format/groundTruth -> True
/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/procedurevrl_hidden/runs/procedurevrl_hidden_full_split1_split1_views16/mstcn_format/splits -> True
/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/procedurevrl_hidden/runs/procedurevrl_hidden_full_split1_split1_views16/mstcn_format/mapping.txt -> True
/conten

## 3. Reproducibility helpers

In [18]:
def set_seed(seed: int = 7):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

set_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)

DEVICE: cuda


## 4. Load mapping and split files

In [19]:
def read_lines(path):
    return [x.strip() for x in Path(path).read_text().splitlines() if x.strip()]


def load_mapping(mapping_path):
    id_to_label = {}
    label_to_id = {}

    for line in read_lines(mapping_path):
        parts = line.split()
        if len(parts) < 2:
            continue

        # Expected MS-TCN format: "0 action_label".
        idx = int(parts[0])
        label = parts[1]
        id_to_label[idx] = label
        label_to_id[label] = idx

    return id_to_label, label_to_id


def load_split(bundle_path):
    video_ids = []
    for line in read_lines(bundle_path):
        video_ids.append(Path(line).stem)
    return video_ids

id_to_label, label_to_id = load_mapping(MAPPING_PATH)
num_classes = len(id_to_label)

train_ids = load_split(TRAIN_SPLIT)
test_ids = load_split(TEST_SPLIT)

ignore_classes = [c for c in CANDIDATE_IGNORE_CLASSES if c in label_to_id]

print("num_classes:", num_classes)
print("train videos:", len(train_ids))
print("test videos:", len(test_ids))
print("ignore_classes:", ignore_classes)
print("first labels:", list(id_to_label.items())[:10])
print("first train ids:", train_ids[:5])
print("first test ids:", test_ids[:5])

assert num_classes > 1
assert len(train_ids) > 0
assert len(test_ids) > 0

num_classes: 48
train videos: 1460
test videos: 252
ignore_classes: ['SIL']
first labels: [(0, 'SIL'), (1, 'pour_cereals'), (2, 'pour_milk'), (3, 'stir_cereals'), (4, 'take_bowl'), (5, 'pour_coffee'), (6, 'take_cup'), (7, 'spoon_sugar'), (8, 'stir_coffee'), (9, 'pour_sugar')]
first train ids: ['P16_cam01_P16_cereals', 'P16_cam01_P16_friedegg', 'P16_cam01_P16_juice', 'P16_cam01_P16_milk', 'P16_cam01_P16_pancake']
first test ids: ['P03_cam01_P03_cereals', 'P03_cam01_P03_coffee', 'P03_cam01_P03_friedegg', 'P03_cam01_P03_milk', 'P03_cam01_P03_salat']


## 5. Verify dataset integrity

In [20]:
def load_labels_as_ids(video_id):
    gt_path = GT_DIR / f"{video_id}.txt"
    labels = read_lines(gt_path)
    unknown = sorted(set(labels) - set(label_to_id.keys()))
    if unknown:
        raise ValueError(f"Unknown labels in {video_id}: {unknown[:10]}")
    return np.array([label_to_id[x] for x in labels], dtype=np.int64), labels

rows = []
unknown_files = []
length_mismatches = []

all_ids = train_ids + test_ids
for video_id in all_ids:
    feat_path = FEATURE_DIR / f"{video_id}.npy"
    gt_path = GT_DIR / f"{video_id}.txt"

    if not feat_path.exists() or not gt_path.exists():
        unknown_files.append(video_id)
        continue

    feat = np.load(feat_path, mmap_mode="r")
    y_ids, y_labels = load_labels_as_ids(video_id)

    if feat.ndim != 2:
        raise ValueError(f"Bad feature rank for {video_id}: {feat.shape}")

    feature_dim, feature_len = int(feat.shape[0]), int(feat.shape[1])
    gt_len = int(len(y_ids))

    if feature_len != gt_len:
        length_mismatches.append((video_id, feature_len, gt_len))

    rows.append({
        "video_id": video_id,
        "split": "train" if video_id in set(train_ids) else "test",
        "feature_dim": feature_dim,
        "feature_len": feature_len,
        "gt_len": gt_len,
        "feat_min": float(np.min(feat)),
        "feat_max": float(np.max(feat)),
        "feat_nan": int(np.isnan(feat).sum()),
    })

df_verify = pd.DataFrame(rows)
verify_path = RESULTS_DIR / "dataset_integrity_check.csv"
df_verify.to_csv(verify_path, index=False)

display(df_verify.head())
print("rows:", len(df_verify))
print("missing files:", len(unknown_files))
print("length mismatches:", len(length_mismatches))
print("feature_dim unique:", sorted(df_verify["feature_dim"].unique().tolist()))
print("feature_len unique:", sorted(df_verify["feature_len"].unique().tolist()))
print("total NaNs:", int(df_verify["feat_nan"].sum()))
print("saved:", verify_path)

assert len(unknown_files) == 0, unknown_files[:10]
assert len(length_mismatches) == 0, length_mismatches[:10]
assert int(df_verify["feat_nan"].sum()) == 0
assert sorted(df_verify["feature_dim"].unique().tolist()) == [512]
assert sorted(df_verify["feature_len"].unique().tolist()) == [16]

,video_id,split,feature_dim,feature_len,gt_len,feat_min,feat_max,feat_nan
0,P16_cam01_P16_cereals,train,512,16,16,-2.339643,4.710883,0
1,P16_cam01_P16_friedegg,train,512,16,16,-2.580863,4.759951,0
2,P16_cam01_P16_juice,train,512,16,16,-2.546146,4.696700,0
3,P16_cam01_P16_milk,train,512,16,16,-2.237515,4.861353,0
4,P16_cam01_P16_pancake,train,512,16,16,-2.616997,4.894865,0


rows: 1712
missing files: 0
length mismatches: 0
feature_dim unique: [512]
feature_len unique: [16]
total NaNs: 0
saved: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/procedurevrl_hidden/runs/mstcn_hidden_baseline_split1/results/dataset_integrity_check.csv


## 6. Compute train feature normalization statistics

In [21]:
def compute_train_stats(video_ids):
    total_sum = None
    total_sumsq = None
    total_count = 0

    for video_id in video_ids:
        x = np.load(FEATURE_DIR / f"{video_id}.npy").astype(np.float64)  # [D, T]
        if total_sum is None:
            total_sum = np.zeros((x.shape[0],), dtype=np.float64)
            total_sumsq = np.zeros((x.shape[0],), dtype=np.float64)

        total_sum += x.sum(axis=1)
        total_sumsq += (x ** 2).sum(axis=1)
        total_count += x.shape[1]

    mean = total_sum / total_count
    var = total_sumsq / total_count - mean ** 2
    var = np.maximum(var, 1e-12)
    std = np.sqrt(var)
    return mean.astype(np.float32), std.astype(np.float32)

if STANDARDIZE_FEATURES:
    train_mean, train_std = compute_train_stats(train_ids)
else:
    train_mean = np.zeros((512,), dtype=np.float32)
    train_std = np.ones((512,), dtype=np.float32)

np.save(RESULTS_DIR / "train_feature_mean.npy", train_mean)
np.save(RESULTS_DIR / "train_feature_std.npy", train_std)

print("standardize:", STANDARDIZE_FEATURES)
print("mean shape:", train_mean.shape)
print("std shape:", train_std.shape)
print("mean min/max:", float(train_mean.min()), float(train_mean.max()))
print("std min/max:", float(train_std.min()), float(train_std.max()))

standardize: True
mean shape: (512,)
std shape: (512,)
mean min/max: -2.1265416145324707 4.203184127807617
std min/max: 0.20662075281143188 0.736768364906311


## 7. Dataset and DataLoader

In [22]:
class BreakfastHiddenDataset(Dataset):
    def __init__(self, video_ids, feature_dir, gt_dir, label_to_id, mean=None, std=None):
        self.video_ids = list(video_ids)
        self.feature_dir = Path(feature_dir)
        self.gt_dir = Path(gt_dir)
        self.label_to_id = dict(label_to_id)
        self.mean = mean
        self.std = std

    def __len__(self):
        return len(self.video_ids)

    def __getitem__(self, idx):
        video_id = self.video_ids[idx]

        x = np.load(self.feature_dir / f"{video_id}.npy").astype(np.float32)  # [D, T]
        labels = read_lines(self.gt_dir / f"{video_id}.txt")
        y = np.array([self.label_to_id[z] for z in labels], dtype=np.int64)  # [T]

        if self.mean is not None and self.std is not None:
            x = (x - self.mean[:, None]) / (self.std[:, None] + 1e-8)

        return {
            "video_id": video_id,
            "features": torch.from_numpy(x),
            "labels": torch.from_numpy(y),
        }

train_dataset = BreakfastHiddenDataset(train_ids, FEATURE_DIR, GT_DIR, label_to_id, train_mean, train_std)
test_dataset = BreakfastHiddenDataset(test_ids, FEATURE_DIR, GT_DIR, label_to_id, train_mean, train_std)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

batch = next(iter(train_loader))
print("batch video ids:", batch["video_id"][:3])
print("features:", batch["features"].shape)
print("labels:", batch["labels"].shape)
print("feature finite:", torch.isfinite(batch["features"]).all().item())

assert batch["features"].shape[1:] == (512, 16)
assert batch["labels"].shape[1:] == (16,)

batch video ids: ['P40_webcam01_P40_tea', 'P25_webcam02_P25_tea', 'P16_webcam02_P16_salat']
features: torch.Size([64, 512, 16])
labels: torch.Size([64, 16])
feature finite: True


## 8. MS-TCN-style model

In [23]:
class DilatedResidualLayer(nn.Module):
    def __init__(self, dilation, channels, dropout):
        super().__init__()
        self.conv_dilated = nn.Conv1d(
            channels,
            channels,
            kernel_size=3,
            padding=dilation,
            dilation=dilation,
        )
        self.conv_1x1 = nn.Conv1d(channels, channels, kernel_size=1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = F.relu(self.conv_dilated(x))
        out = self.conv_1x1(out)
        out = self.dropout(out)
        return x + out


class SingleStageModel(nn.Module):
    def __init__(self, num_layers, num_f_maps, input_dim, num_classes, dropout):
        super().__init__()
        self.conv_in = nn.Conv1d(input_dim, num_f_maps, kernel_size=1)
        self.layers = nn.ModuleList([
            DilatedResidualLayer(2 ** i, num_f_maps, dropout)
            for i in range(num_layers)
        ])
        self.conv_out = nn.Conv1d(num_f_maps, num_classes, kernel_size=1)

    def forward(self, x):
        out = self.conv_in(x)
        for layer in self.layers:
            out = layer(out)
        out = self.conv_out(out)
        return out


class MultiStageModel(nn.Module):
    def __init__(self, num_stages, num_layers, num_f_maps, input_dim, num_classes, dropout):
        super().__init__()
        assert num_stages >= 1
        self.stage1 = SingleStageModel(num_layers, num_f_maps, input_dim, num_classes, dropout)
        self.stages = nn.ModuleList([
            SingleStageModel(num_layers, num_f_maps, num_classes, num_classes, dropout)
            for _ in range(num_stages - 1)
        ])

    def forward(self, x):
        outputs = []
        out = self.stage1(x)
        outputs.append(out)
        for stage in self.stages:
            out = stage(F.softmax(out, dim=1))
            outputs.append(out)
        return torch.stack(outputs, dim=0)  # [S, B, C, T]

model = MultiStageModel(
    num_stages=NUM_STAGES,
    num_layers=NUM_LAYERS,
    num_f_maps=NUM_F_MAPS,
    input_dim=512,
    num_classes=num_classes,
    dropout=DROPOUT,
).to(DEVICE)

num_params = sum(p.numel() for p in model.parameters())
num_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(model)
print("num_params:", num_params)
print("num_trainable:", num_trainable)

with torch.no_grad():
    x = batch["features"].to(DEVICE)
    yhat = model(x)
    print("model output:", yhat.shape)
    assert yhat.shape == (NUM_STAGES, x.shape[0], num_classes, 16)

MultiStageModel(
  (stage1): SingleStageModel(
    (conv_in): Conv1d(512, 64, kernel_size=(1,), stride=(1,))
    (layers): ModuleList(
      (0): DilatedResidualLayer(
        (conv_dilated): Conv1d(64, 64, kernel_size=(3,), stride=(1,), padding=(1,))
        (conv_1x1): Conv1d(64, 64, kernel_size=(1,), stride=(1,))
        (dropout): Dropout(p=0.5, inplace=False)
      )
      (1): DilatedResidualLayer(
        (conv_dilated): Conv1d(64, 64, kernel_size=(3,), stride=(1,), padding=(2,), dilation=(2,))
        (conv_1x1): Conv1d(64, 64, kernel_size=(1,), stride=(1,))
        (dropout): Dropout(p=0.5, inplace=False)
      )
      (2): DilatedResidualLayer(
        (conv_dilated): Conv1d(64, 64, kernel_size=(3,), stride=(1,), padding=(4,), dilation=(4,))
        (conv_1x1): Conv1d(64, 64, kernel_size=(1,), stride=(1,))
        (dropout): Dropout(p=0.5, inplace=False)
      )
      (3): DilatedResidualLayer(
        (conv_dilated): Conv1d(64, 64, kernel_size=(3,), stride=(1,), padding=(8,)

## 9. Evaluation metrics

In [24]:
def collapse_segments(frame_labels, bg_classes=None):
    bg_classes = set(bg_classes or [])
    labels = []
    starts = []
    ends = []

    last = None
    for i, lab in enumerate(frame_labels):
        if lab in bg_classes:
            if last is not None:
                ends.append(i)
                last = None
            continue

        if lab != last:
            if last is not None:
                ends.append(i)
            labels.append(lab)
            starts.append(i)
            last = lab

    if last is not None:
        ends.append(len(frame_labels))

    return labels, starts, ends


def levenshtein_distance(pred, gt):
    m, n = len(pred), len(gt)
    dp = np.zeros((m + 1, n + 1), dtype=np.int32)
    for i in range(m + 1):
        dp[i, 0] = i
    for j in range(n + 1):
        dp[0, j] = j

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            cost = 0 if pred[i - 1] == gt[j - 1] else 1
            dp[i, j] = min(
                dp[i - 1, j] + 1,
                dp[i, j - 1] + 1,
                dp[i - 1, j - 1] + cost,
            )
    return int(dp[m, n])


def edit_score_single(pred_labels, gt_labels, bg_classes=None):
    p, _, _ = collapse_segments(pred_labels, bg_classes)
    y, _, _ = collapse_segments(gt_labels, bg_classes)

    if len(p) == 0 and len(y) == 0:
        return 100.0
    if max(len(p), len(y)) == 0:
        return 0.0

    dist = levenshtein_distance(p, y)
    return (1.0 - dist / max(len(p), len(y))) * 100.0


def f_score_single(pred_labels, gt_labels, overlap, bg_classes=None):
    p_label, p_start, p_end = collapse_segments(pred_labels, bg_classes)
    y_label, y_start, y_end = collapse_segments(gt_labels, bg_classes)

    tp = 0
    fp = 0
    hits = np.zeros(len(y_label), dtype=np.float32)

    for j in range(len(p_label)):
        best_iou = 0.0
        best_idx = -1

        for i in range(len(y_label)):
            if p_label[j] != y_label[i]:
                continue

            inter = min(p_end[j], y_end[i]) - max(p_start[j], y_start[i])
            union = max(p_end[j], y_end[i]) - min(p_start[j], y_start[i])
            iou = max(inter, 0) / union if union > 0 else 0.0

            if iou > best_iou:
                best_iou = iou
                best_idx = i

        if best_iou >= overlap and best_idx >= 0 and hits[best_idx] == 0:
            tp += 1
            hits[best_idx] = 1
        else:
            fp += 1

    fn = len(y_label) - int(hits.sum())
    return tp, fp, fn


def compute_metrics(pred_by_video, gt_by_video, id_to_label, bg_classes=None):
    bg_classes = bg_classes or []

    total_correct = 0
    total_frames = 0
    edit_scores = []

    f_stats = {
        0.10: [0, 0, 0],
        0.25: [0, 0, 0],
        0.50: [0, 0, 0],
    }

    for video_id, pred_ids in pred_by_video.items():
        gt_ids = gt_by_video[video_id]
        assert len(pred_ids) == len(gt_ids), (video_id, len(pred_ids), len(gt_ids))

        total_correct += int((pred_ids == gt_ids).sum())
        total_frames += int(len(gt_ids))

        pred_labels = [id_to_label[int(x)] for x in pred_ids]
        gt_labels = [id_to_label[int(x)] for x in gt_ids]

        edit_scores.append(edit_score_single(pred_labels, gt_labels, bg_classes))

        for overlap in f_stats:
            tp, fp, fn = f_score_single(pred_labels, gt_labels, overlap, bg_classes)
            f_stats[overlap][0] += tp
            f_stats[overlap][1] += fp
            f_stats[overlap][2] += fn

    acc = 100.0 * total_correct / max(total_frames, 1)
    edit = float(np.mean(edit_scores)) if edit_scores else 0.0

    out = {
        "acc": acc,
        "edit": edit,
    }

    for overlap, (tp, fp, fn) in f_stats.items():
        precision = tp / max(tp + fp, 1e-8)
        recall = tp / max(tp + fn, 1e-8)
        f1 = 2.0 * precision * recall / max(precision + recall, 1e-8)
        out[f"f1@{int(overlap * 100)}"] = 100.0 * f1

    return out

print("Metric functions ready.")

Metric functions ready.


## 10. Train/evaluate helpers

In [25]:
def mstcn_loss(outputs, targets, smoothing_weight=0.15):
    # outputs: [S, B, C, T]
    # targets: [B, T]
    total = 0.0

    for s in range(outputs.shape[0]):
        logits = outputs[s]
        ce = F.cross_entropy(
            logits.permute(0, 2, 1).reshape(-1, logits.shape[1]),
            targets.reshape(-1),
        )

        log_probs = F.log_softmax(logits, dim=1)
        smooth = F.mse_loss(
            log_probs[:, :, 1:],
            log_probs.detach()[:, :, :-1],
            reduction="none",
        )
        smooth = torch.clamp(smooth, min=0.0, max=16.0).mean()

        total = total + ce + smoothing_weight * smooth

    return total


@torch.no_grad()
def evaluate_model(model, loader, save_predictions=False, pred_dir=None):
    model.eval()
    pred_by_video = {}
    gt_by_video = {}

    if save_predictions:
        pred_dir = Path(pred_dir)
        pred_dir.mkdir(parents=True, exist_ok=True)

    for batch in loader:
        x = batch["features"].to(DEVICE, non_blocking=True)
        y = batch["labels"].cpu().numpy()
        video_ids = list(batch["video_id"])

        outputs = model(x)
        logits = outputs[-1]
        pred = logits.argmax(dim=1).detach().cpu().numpy()  # [B, T]

        for i, video_id in enumerate(video_ids):
            pred_ids = pred[i].astype(np.int64)
            gt_ids = y[i].astype(np.int64)
            pred_by_video[video_id] = pred_ids
            gt_by_video[video_id] = gt_ids

            if save_predictions:
                pred_labels = [id_to_label[int(z)] for z in pred_ids]
                (pred_dir / f"{video_id}.txt").write_text("\n".join(pred_labels) + "\n")

    metrics = compute_metrics(pred_by_video, gt_by_video, id_to_label, bg_classes=ignore_classes)
    return metrics, pred_by_video, gt_by_video


optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

print("Helpers ready.")

Helpers ready.


## 11. Train MS-TCN-style baseline

In [26]:
best_score = -1.0
best_epoch = None
best_metrics = None
history = []

best_model_path = MODEL_DIR / "best_model.pt"
last_model_path = MODEL_DIR / "last_model.pt"

start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_losses = []

    for batch in train_loader:
        x = batch["features"].to(DEVICE, non_blocking=True)
        y = batch["labels"].to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        outputs = model(x)
        loss = mstcn_loss(outputs, y, smoothing_weight=SMOOTHING_LOSS_WEIGHT)
        loss.backward()

        if GRAD_CLIP is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)

        optimizer.step()
        epoch_losses.append(float(loss.detach().cpu().item()))

    scheduler.step()

    row = {
        "epoch": epoch,
        "train_loss": float(np.mean(epoch_losses)),
        "lr": float(scheduler.get_last_lr()[0]),
    }

    do_eval = epoch == 1 or epoch % EVAL_EVERY == 0 or epoch == EPOCHS
    if do_eval:
        metrics, _, _ = evaluate_model(model, test_loader, save_predictions=False)
        row.update(metrics)

        # Exploratory selection score. We still also save the final epoch separately.
        score = metrics["f1@25"] + 0.01 * metrics["edit"]
        if score > best_score:
            best_score = score
            best_epoch = epoch
            best_metrics = dict(metrics)
            torch.save({
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "metrics": metrics,
                "config": {
                    "num_stages": NUM_STAGES,
                    "num_layers": NUM_LAYERS,
                    "num_f_maps": NUM_F_MAPS,
                    "dropout": DROPOUT,
                    "input_dim": 512,
                    "num_classes": num_classes,
                    "standardize_features": STANDARDIZE_FEATURES,
                },
            }, best_model_path)

        print(
            f"epoch {epoch:03d} "
            f"loss={row['train_loss']:.4f} "
            f"acc={metrics['acc']:.2f} "
            f"edit={metrics['edit']:.2f} "
            f"f1@10={metrics['f1@10']:.2f} "
            f"f1@25={metrics['f1@25']:.2f} "
            f"f1@50={metrics['f1@50']:.2f}"
        )
    else:
        print(f"epoch {epoch:03d} loss={row['train_loss']:.4f}")

    history.append(row)

elapsed_min = (time.time() - start_time) / 60.0

torch.save({
    "epoch": EPOCHS,
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
}, last_model_path)

history_df = pd.DataFrame(history)
history_path = RESULTS_DIR / "training_history.csv"
history_df.to_csv(history_path, index=False)

print("elapsed minutes:", elapsed_min)
print("best_epoch:", best_epoch)
print("best_metrics:", best_metrics)
print("saved best:", best_model_path)
print("saved last:", last_model_path)
print("saved history:", history_path)

display(history_df.tail())

epoch 001 loss=14.3332 acc=11.68 edit=0.00 f1@10=0.00 f1@25=0.00 f1@50=0.00
epoch 002 loss=12.7719
epoch 003 loss=12.0664
epoch 004 loss=11.3421
epoch 005 loss=10.5891 acc=18.73 edit=11.89 f1@10=12.65 f1@25=11.11 f1@50=5.79
epoch 006 loss=9.9892
epoch 007 loss=9.3732
epoch 008 loss=8.7613
epoch 009 loss=8.1700
epoch 010 loss=7.7613 acc=42.93 edit=35.75 f1@10=40.28 f1@25=36.68 f1@50=22.05
epoch 011 loss=7.3827
epoch 012 loss=7.0884
epoch 013 loss=6.8549
epoch 014 loss=6.7003
epoch 015 loss=6.5397 acc=46.97 edit=43.91 f1@10=48.51 f1@25=44.82 f1@50=29.63
epoch 016 loss=6.4014
epoch 017 loss=6.2671
epoch 018 loss=6.1750
epoch 019 loss=6.0268
epoch 020 loss=5.9655 acc=51.09 edit=49.46 f1@10=53.06 f1@25=50.48 f1@50=37.58
epoch 021 loss=5.8273
epoch 022 loss=5.7173
epoch 023 loss=5.6458
epoch 024 loss=5.5839
epoch 025 loss=5.4675 acc=53.55 edit=50.92 f1@10=54.17 f1@25=50.95 f1@50=40.27
epoch 026 loss=5.3736
epoch 027 loss=5.3344
epoch 028 loss=5.2754
epoch 029 loss=5.2397
epoch 030 loss=5.153

,epoch,train_loss,lr,acc,edit,f1@10,f1@25,f1@50
115,116,3.595629,1.369526e-06,NaN,NaN,NaN,NaN,NaN
116,117,3.597678,7.706666e-07,NaN,NaN,NaN,NaN,NaN
117,118,3.582416,3.426163e-07,NaN,NaN,NaN,NaN,NaN
118,119,3.584037,8.566876e-08,NaN,NaN,NaN,NaN,NaN
119,120,3.606218,0.000000e+00,58.804563,56.274408,57.688229,56.521739,45.705196


## 12. Evaluate best checkpoint and save predictions

In [27]:
# Reload best checkpoint for final reported exploratory result.
checkpoint = torch.load(best_model_path, map_location=DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
model.to(DEVICE)
model.eval()

best_test_pred_dir = PRED_DIR / "best_checkpoint_test_predictions"
best_metrics, pred_by_video, gt_by_video = evaluate_model(
    model,
    test_loader,
    save_predictions=True,
    pred_dir=best_test_pred_dir,
)

best_metrics_rounded = {k: round(float(v), 2) for k, v in best_metrics.items()}

print("best checkpoint epoch:", checkpoint["epoch"])
print("best checkpoint metrics:")
print(json.dumps(best_metrics_rounded, indent=2))
print("prediction files:", len(list(best_test_pred_dir.glob("*.txt"))))
print("prediction dir:", best_test_pred_dir)

assert len(list(best_test_pred_dir.glob("*.txt"))) == len(test_ids)

best checkpoint epoch: 55
best checkpoint metrics:
{
  "acc": 58.04,
  "edit": 55.57,
  "f1@10": 58.58,
  "f1@25": 57.73,
  "f1@50": 45.76
}
prediction files: 252
prediction dir: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/procedurevrl_hidden/runs/mstcn_hidden_baseline_split1/predictions/best_checkpoint_test_predictions


## 13. Save final result tables

In [28]:
final_metrics_row = {
    "experiment": "procedurevrl_hidden_mstcn_visual_only",
    "features": "ProcedureVRL hidden video embeddings",
    "feature_shape": "[512, 16]",
    "model": "MS-TCN-style visual-only",
    "split": "Breakfast split 1",
    "selection": "best checkpoint by exploratory test F1@25/Edit score",
    "best_epoch": int(checkpoint["epoch"]),
    "acc": best_metrics_rounded["acc"],
    "edit": best_metrics_rounded["edit"],
    "f1@10": best_metrics_rounded["f1@10"],
    "f1@25": best_metrics_rounded["f1@25"],
    "f1@50": best_metrics_rounded["f1@50"],
    "note": (
        "Visual-only baseline on hidden 512-dimensional ProcedureVRL video embeddings. "
        "Temporal length is still 16 clips per video."
    ),
}

final_metrics_df = pd.DataFrame([final_metrics_row])
final_metrics_path = RESULTS_DIR / "final_metrics.csv"
final_metrics_md_path = RESULTS_DIR / "final_metrics.md"

final_metrics_df.to_csv(final_metrics_path, index=False)
final_metrics_md_path.write_text(final_metrics_df.to_markdown(index=False))

display(final_metrics_df)
print("saved CSV:", final_metrics_path)
print("saved Markdown:", final_metrics_md_path)

,experiment,features,feature_shape,model,split,selection,best_epoch,acc,edit,f1@10,f1@25,f1@50,note
0,procedurevrl_hidden_mstcn_visual_only,ProcedureVRL hidden video embeddings,"[512, 16]",MS-TCN-style visual-only,Breakfast split 1,best checkpoint by exploratory test F1@25/Edit...,55,58.04,55.57,58.58,57.73,45.76,Visual-only baseline on hidden 512-dimensional...


saved CSV: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/procedurevrl_hidden/runs/mstcn_hidden_baseline_split1/results/final_metrics.csv
saved Markdown: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/procedurevrl_hidden/runs/mstcn_hidden_baseline_split1/results/final_metrics.md


## 14. Comparison with previous runs

In [29]:
comparison_rows = [
    {
        "experiment": "I3D baseline",
        "features": "I3D",
        "model": "Official MS-TCN visual-only, 30 epochs",
        "feature_shape": "[2048, T]",
        "temporal_resolution": "frame/segment-level variable T",
        "acc": 55.38,
        "edit": 44.97,
        "f1@10": 39.05,
        "f1@25": 34.80,
        "f1@50": 25.25,
        "note": "Previous official-style visual-only baseline.",
    },
    {
        "experiment": "I3D + CLIP KD proof-of-concept",
        "features": "I3D + CLIP text teacher",
        "model": "Video-only KD student, lambda=0.05",
        "feature_shape": "[2048, T]",
        "temporal_resolution": "frame/segment-level variable T",
        "acc": 70.98,
        "edit": 58.53,
        "f1@10": 50.16,
        "f1@25": 46.86,
        "f1@50": 38.33,
        "note": "Useful proof-of-concept, but visual/text spaces are not aligned.",
    },
    {
        "experiment": "ProcedureVRL coarse baseline",
        "features": "ProcedureVRL coarse output-level features",
        "model": "MS-TCN-style visual-only",
        "feature_shape": "[9871, 16]",
        "temporal_resolution": "16 clips per video",
        "acc": 48.91,
        "edit": 49.15,
        "f1@10": 55.46,
        "f1@25": 54.38,
        "f1@50": 42.61,
        "note": "Pipeline validation on coarse output-level features.",
    },
    {
        "experiment": "ProcedureVRL hidden baseline",
        "features": "ProcedureVRL hidden video embeddings",
        "model": "MS-TCN-style visual-only",
        "feature_shape": "[512, 16]",
        "temporal_resolution": "16 clips per video",
        "acc": best_metrics_rounded["acc"],
        "edit": best_metrics_rounded["edit"],
        "f1@10": best_metrics_rounded["f1@10"],
        "f1@25": best_metrics_rounded["f1@25"],
        "f1@50": best_metrics_rounded["f1@50"],
        "note": "Stronger ProcedureVRL representation for the next text-teacher/KD step.",
    },
]

comparison_df = pd.DataFrame(comparison_rows)
comparison_path = RESULTS_DIR / "comparison_with_previous_runs.csv"
comparison_md_path = RESULTS_DIR / "comparison_with_previous_runs.md"

comparison_df.to_csv(comparison_path, index=False)
comparison_md_path.write_text(comparison_df.to_markdown(index=False))

display(comparison_df)
print("saved CSV:", comparison_path)
print("saved Markdown:", comparison_md_path)

print("\nImportant caveat:")
print(
    "ProcedureVRL coarse and hidden runs use temporal length 16 per video. "
    "They are useful for ProcedureVRL-pipeline comparison, but they are not fully fair "
    "frame-level comparisons against I3D features with variable T."
)

,experiment,features,model,feature_shape,temporal_resolution,acc,edit,f1@10,f1@25,f1@50,note
0,I3D baseline,I3D,"Official MS-TCN visual-only, 30 epochs","[2048, T]",frame/segment-level variable T,55.38,44.97,39.05,34.80,25.25,Previous official-style visual-only baseline.
1,I3D + CLIP KD proof-of-concept,I3D + CLIP text teacher,"Video-only KD student, lambda=0.05","[2048, T]",frame/segment-level variable T,70.98,58.53,50.16,46.86,38.33,"Useful proof-of-concept, but visual/text space..."
2,ProcedureVRL coarse baseline,ProcedureVRL coarse output-level features,MS-TCN-style visual-only,"[9871, 16]",16 clips per video,48.91,49.15,55.46,54.38,42.61,Pipeline validation on coarse output-level fea...
3,ProcedureVRL hidden baseline,ProcedureVRL hidden video embeddings,MS-TCN-style visual-only,"[512, 16]",16 clips per video,58.04,55.57,58.58,57.73,45.76,Stronger ProcedureVRL representation for the n...


saved CSV: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/procedurevrl_hidden/runs/mstcn_hidden_baseline_split1/results/comparison_with_previous_runs.csv
saved Markdown: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/procedurevrl_hidden/runs/mstcn_hidden_baseline_split1/results/comparison_with_previous_runs.md

Important caveat:
ProcedureVRL coarse and hidden runs use temporal length 16 per video. They are useful for ProcedureVRL-pipeline comparison, but they are not fully fair frame-level comparisons against I3D features with variable T.


## 15. Final summary

In [30]:
summary = {
    "status": "completed",
    "experiment": "mstcn_hidden_procedurevrl_baseline_breakfast_split1",
    "input_dataset_root": str(DATA_ROOT),
    "output_root": str(OUT_ROOT),
    "features": "ProcedureVRL hidden video embeddings",
    "feature_shape_per_video": [512, 16],
    "num_train_videos": len(train_ids),
    "num_test_videos": len(test_ids),
    "num_classes": num_classes,
    "ignore_classes": ignore_classes,
    "model": {
        "type": "MS-TCN-style MultiStageModel",
        "num_stages": NUM_STAGES,
        "num_layers": NUM_LAYERS,
        "num_f_maps": NUM_F_MAPS,
        "dropout": DROPOUT,
        "num_params": int(num_params),
    },
    "training": {
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "smoothing_loss_weight": SMOOTHING_LOSS_WEIGHT,
        "standardize_features": STANDARDIZE_FEATURES,
    },
    "best_checkpoint_epoch": int(checkpoint["epoch"]),
    "best_checkpoint_metrics": best_metrics_rounded,
    "paths": {
        "best_model": str(best_model_path),
        "last_model": str(last_model_path),
        "training_history": str(history_path),
        "final_metrics_csv": str(final_metrics_path),
        "comparison_csv": str(comparison_path),
        "predictions": str(best_test_pred_dir),
    },
    "note": (
        "This completes the visual-only MS-TCN-style baseline on hidden ProcedureVRL embeddings. "
        "The next step is the text-teacher / video-only-student KD experiment in the same feature space."
    ),
}

summary_path = RESULTS_DIR / "final_summary.json"
summary_path.write_text(json.dumps(summary, indent=2))

print(json.dumps(summary, indent=2))
print("\nSaved:", summary_path)

{
  "status": "completed",
  "experiment": "mstcn_hidden_procedurevrl_baseline_breakfast_split1",
  "input_dataset_root": "/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/procedurevrl_hidden/runs/procedurevrl_hidden_full_split1_split1_views16/mstcn_format",
  "output_root": "/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/procedurevrl_hidden/runs/mstcn_hidden_baseline_split1",
  "features": "ProcedureVRL hidden video embeddings",
  "feature_shape_per_video": [
    512,
    16
  ],
  "num_train_videos": 1460,
  "num_test_videos": 252,
  "num_classes": 48,
  "ignore_classes": [
    "SIL"
  ],
  "model": {
    "type": "MS-TCN-style MultiStageModel",
    "num_stages": 4,
    "num_layers": 6,
    "num_f_maps": 64,
    "dropout": 0.5,
    "num_params": 451008
  },
  "training": {
    "epochs": 120,
    "batch_size": 64,
    "learning_rate": 0.0005,
    "weight_decay": 0.0001,
    "smoothing_loss_weight": 0.15,
    "standardize_features": true
  },
  "b